In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Problem statement
Amazon is expanding its AI-powered customer service ecosystem and enterprise AI services through cloud infrastructure, automation, AI assistants, and connected service platforms. This expansion creates opportunities for efficiency and market growth, but it also introduces strategic, operational, technology, information security, financial, compliance, and governance risks. This notebook uses the Amazon risk register to evaluate risk severity based on likelihood and impact scores, classify risks into low, medium, and high categories, and produce a risk heat map consistent with the course risk assessment requirements.

In [ ]:
# Load the Amazon Risk Register workbook
# Keep the Excel file in the same folder as this notebook when running locally.

file_path = "ALY6130_Risk Register_Group3_Amazon.xlsx"

# Read the Risk Register sheet from the workbook
# Only the required columns are selected because the original workbook contains multiple sections.
df_risk_register = pd.read_excel(
    file_path,
    sheet_name="RiskRegister",
    usecols=["Risk #", "The Risk of/That", "Likelihood Score", "Impact Score"]
)

# Preview the first few rows
df_risk_register.head()

In [ ]:
# Select and rename relevant columns
risk_df = df_risk_register[['Risk #', 'The Risk of/That', 'Likelihood Score', 'Impact Score']].dropna()
risk_df.columns = ['Risk_ID', 'Risk_Description', 'Likelihood_Score', 'Impact_Score']

# Make sure scores are numeric
risk_df['Risk_ID'] = risk_df['Risk_ID'].astype(int)
risk_df['Likelihood_Score'] = pd.to_numeric(risk_df['Likelihood_Score'], errors='coerce')
risk_df['Impact_Score'] = pd.to_numeric(risk_df['Impact_Score'], errors='coerce')
risk_df = risk_df.dropna(subset=['Likelihood_Score', 'Impact_Score'])

# Calculate total risk score
risk_df['Risk_Score'] = risk_df['Likelihood_Score'] * risk_df['Impact_Score']

# Define severity label based on likelihood and impact scores
# This follows the teacher notebook structure: High risk requires both high likelihood and high impact.
def classify_severity(row):
    if row['Likelihood_Score'] >= 8 and row['Impact_Score'] >= 5:
        return 'High'
    elif row['Likelihood_Score'] >= 5 or row['Impact_Score'] >= 3:
        return 'Medium'
    else:
        return 'Low'

# Apply function to classify severity
risk_df['Severity_Label'] = risk_df.apply(classify_severity, axis=1)

# Create label used in charts
risk_df['Label'] = 'R' + risk_df['Risk_ID'].astype(str) + ' (' + risk_df['Severity_Label'] + ')'

# View cleaned data
risk_df.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# This model uses a Random Forest classifier to predict risk severity levels based on likelihood and impact scores.
# The model is used as a supporting analytical exercise; the main ERM scoring still comes from the Risk Register.
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

In [ ]:
# Re-select features and target: defines the feature matrix (X) using Likelihood and Impact scores,
# and creates the target variable (y) using the rule-based severity labels.
X = risk_df[['Likelihood_Score', 'Impact_Score']]
y = risk_df['Severity_Label']

# Check class distribution before model evaluation
print("Severity class distribution:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Use stratified cross-validation because the dataset is small but contains multiple severity classes.
# This gives a more stable evaluation than a single train-test split.
n_splits = min(5, y.value_counts().min())
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print(f"Mean accuracy ({n_splits}-fold CV): {scores.mean():.2f}")
print("Accuracy for each fold:", scores)

### Note:
The model accuracy should be interpreted with caution because the severity labels are derived from the same likelihood and impact scores used as model features. The Random Forest model is included to follow the analytical notebook structure, but the main risk assessment decision should still be based on the Risk Register, scoring logic, and ERM interpretation.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

# Cross-validated predictions
y_pred_cv = cross_val_predict(model, X, y, cv=cv)

# Classification metrics
print("Classification Report:")
print(classification_report(y, y_pred_cv))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y, y_pred_cv, labels=['Low', 'Medium', 'High']))

The classification results show how consistently the rule-based severity categories can be reproduced from likelihood and impact scores. Since the target severity is based on these two scores, high accuracy is expected and should not be over-interpreted as an independent predictive finding.

In [ ]:
# Create severity map for risk heat map
severity_map = risk_df.copy()

# Map severity to numeric scale
severity_levels = {'Low': 1, 'Medium': 2, 'High': 3}
severity_map['Severity_Num'] = severity_map['Severity_Label'].map(severity_levels)

# Pivot table with most severe risk per likelihood-impact cell
pivot_severity = severity_map.groupby(['Likelihood_Score', 'Impact_Score'])['Severity_Num'].max().unstack()
pivot_severity = pivot_severity.sort_index(ascending=False)
pivot_severity = pivot_severity[pivot_severity.columns.sort_values()]

# Annotation matrix with risk IDs and severity labels
label_matrix = severity_map.groupby(['Likelihood_Score', 'Impact_Score'])['Label']                            .apply(lambda x: '
'.join(x)).unstack(fill_value='')
label_matrix = label_matrix.sort_index(ascending=False)
label_matrix = label_matrix[label_matrix.columns.sort_values()]

# Make blank cells white
cmap = plt.cm.get_cmap('RdYlGn_r').copy()
cmap.set_bad(color='white')

# Plot severity heatmap with annotations using the same teacher-style format
plt.figure(figsize=(14, 8))
ax = sns.heatmap(
    pivot_severity,
    annot=label_matrix,
    fmt='',
    cmap=cmap,
    vmin=1,
    vmax=3,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Severity Level (1=Low, 3=High)'},
    annot_kws={'fontsize': 8, 'ha': 'center', 'va': 'center'}
)

plt.title("Annotated Risk Heatmap by Severity (Red = High Risk)", fontsize=14)
plt.xlabel("Impact Score", fontsize=11)
plt.ylabel("Likelihood Score", fontsize=11)
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

# Reuse model and labels
y_pred_cv = cross_val_predict(model, X, y, cv=cv)
conf_matrix = confusion_matrix(y, y_pred_cv, labels=['Low', 'Medium', 'High'])

# Plot confusion matrix as heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Low', 'Medium', 'High'],
    yticklabels=['Low', 'Medium', 'High']
)
plt.title("Confusion Matrix (Cross-Validated)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

### Interpretation
The matrix shows how well the model predicts risk severity levels based on likelihood and impact scores. In this analysis, the model is mainly used to support the notebook workflow. The ERM interpretation should focus on the risk register, severity thresholds, and heat map distribution.

In [ ]:
# Summary of risks by severity level
severity_summary = risk_df.groupby('Severity_Label').agg(
    Number_of_Risks=('Risk_ID', 'count'),
    Average_Likelihood=('Likelihood_Score', 'mean'),
    Average_Impact=('Impact_Score', 'mean'),
    Average_Risk_Score=('Risk_Score', 'mean')
).reindex(['Low', 'Medium', 'High'])

severity_summary

In [ ]:
# Top 10 risks by total risk score
top_10_risks = risk_df.sort_values('Risk_Score', ascending=False).head(10)

top_10_risks[['Risk_ID', 'Risk_Description', 'Likelihood_Score', 'Impact_Score', 'Risk_Score', 'Severity_Label']]